In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.read_csv("telecom_master.csv")
implied = df["total_charges"] / df["tenure_months"]
print("Correlation with arpu:", round(implied.corr(df["arpu"]), 4))
print(df[["arpu"]].join(implied.rename("implied_arpu")).head())

In [ ]:
df["tenure_band"] = pd.cut(
    df["tenure_months"],
    bins=[0, 6, 18, 36, 72],
    labels=["0-6m", "7-18m", "19-36m", "37m+"],
)
df["data_per_voice"] = df["avg_monthly_gb"] / df["avg_voice_min"].clip(lower=1)
df["complaints_per_yr"] = (
    df["complaints_6m"] * 2 / (df["tenure_months"] / 12).clip(lower=0.5)
)
df["is_heavy_data"] = (
    df["avg_monthly_gb"] > df["avg_monthly_gb"].quantile(0.75)
).astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

drop = ["arpu", "customer_id", "total_charges", "churn"]
X = df.drop(columns=drop)
y = df["arpu"]
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
pre = ColumnTransformer(
    [
        (
            "n",
            Pipeline(
                [("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]
            ),
            num,
        ),
        (
            "c",
            Pipeline(
                [
                    ("i", SimpleImputer(strategy="most_frequent")),
                    ("o", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat,
        ),
    ]
)

for name, reg in [
    ("Linear", LinearRegression()),
    ("Random forest", RandomForestRegressor(n_estimators=300, random_state=42)),
]:
    m = Pipeline([("pre", pre), ("reg", reg)]).fit(X_tr, y_tr)
    p = m.predict(X_te)
    print(
        f"{name:14s} RMSE {mean_squared_error(y_te, p)**0.5:7.2f}"
        f"  MAE {mean_absolute_error(y_te, p):7.2f}  R2 {r2_score(y_te, p):.3f}"
    )

In [ ]:
model = Pipeline(
    [("pre", pre), ("reg", RandomForestRegressor(n_estimators=300, random_state=42))]
).fit(X_tr, y_tr)
pred = model.predict(X_te)
resid = y_te - pred

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(pred, resid, s=8, alpha=0.4)
ax[0].axhline(0, color="crimson")
ax[0].set_xlabel("Predicted ARPU")
ax[0].set_ylabel("Residual")
ax[1].scatter(X_te["tenure_months"], resid, s=8, alpha=0.4)
ax[1].axhline(0, color="crimson")
ax[1].set_xlabel("Tenure (months)")
plt.tight_layout()
plt.show()

In [ ]:
seg = X_te.assign(resid=resid.values)
print(seg.groupby("region")["resid"].agg(["mean", "std", "count"]).round(2))
print(seg.groupby("plan_type")["resid"].agg(["mean", "std", "count"]).round(2))

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(model, X_te, y_te, n_repeats=8, random_state=1)
names = X_te.columns
pd.Series(imp.importances_mean, index=names).sort_values(ascending=False).head(8).round(
    4
)